In [1]:
# NS
#optiver-ver (for ensemble)

#optiver-ver-nn-modeling/inference-v1
#optiver-ver-rnn-modeling/inference-v1
#optiver-ver-lgb-modeling/inference-v1


import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from scipy.stats import hmean

from sklearn.ensemble import HistGradientBoostingRegressor
import itertools
import pickle
import joblib
from itertools import combinations
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, Flatten, Concatenate, GaussianNoise
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.optimizers.schedules import ExponentialDecay
from tensorflow.keras.layers import concatenate,Dropout
import pickle
from tensorflow.keras.models import load_model
import os 
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras.metrics import MeanAbsoluteError
from tensorflow.keras.callbacks import Callback
import random
from tensorflow.keras.layers import Input, Embedding, Lambda, Reshape, LSTM, Dense, BatchNormalization, Dropout, concatenate
from tensorflow.keras import backend as K
from tensorflow.keras.layers import ZeroPadding1D
from tensorflow.keras.layers import Conv1D
from tensorflow.keras.layers import RepeatVector

from tensorflow.keras.layers import Input, Embedding, Lambda, Reshape, LSTM, Dense, BatchNormalization, Dropout, concatenate
from tensorflow.keras import backend as K
from tensorflow.keras.layers import ZeroPadding1D, Activation
from tensorflow.keras.layers import Conv1D
from tensorflow.keras.layers import RepeatVector
from tensorflow.keras.layers import MaxPooling1D, AveragePooling1D
from tensorflow.keras.layers import Add

#---------------------------------------------------------------------- setup dashboard ------------------------------------------------------------

kaggle              = False
is_inference        = False
load_models         = False
run_pipeline        = True
train_models        = True
is_lgb              = True    #1
is_nn               = False    #2
is_rnn              = False   #3
simulation          = False
online_learning     = False

manage_memory       = False
memory_threshold    = 24576  #24GB

ens_models          = [1.0,1.0,1.0]

#public-validation
# dates_train = [0,390]
# dates_test = [391,480]

#full-inference
dates_train = [0,480]
dates_test = [-1,-1]

num_models ={'lgb':1,'nn':1,'rnn':1} 



train_path = r"C:\Users\cwang\Desktop\Kaggle_optiver\train.csv"
models_path = r"/kaggle/input/optiver-rnn-just-imb-models/"

#---------------------------------------------------------------------- setup dashboard ------------------------------------------------------------


In [2]:
#@title params dashboard
lgb_params = {
            'learning_rate'     : 0.005,  #0.005,0.05
            'max_depth'         : 14, #14
            'n_estimators'      : 5000,
            'num_leaves'        : 1023,    #511,31,1023
            'objective'         : 'mae',
            'subsample'         : .2, 
            'colsample_bytree'  : .3,
            'num_threads'       : 32,
            'device'            : 'gpu',
            # 'reg_alpha'         : 0.1,
            # 'reg_lambda'        : 4,
        }

In [3]:
pd.set_option('mode.chained_assignment', None)


def convert_price_cols_float32(df):

    # Columns containing 'price'
    price_columns = [col for col in df.columns if 'price' in col]
    df[price_columns] = df[price_columns].astype('float32')

    # Columns containing 'wap'
    wap_columns = [col for col in df.columns if 'wap' in col]
    df[wap_columns] = df[wap_columns].astype('float32')

    return df

train = pd.read_csv(train_path).drop(['row_id', 'time_id'], axis = 1)
nan_count = train['target'].isna().sum()
print(f"The 'target' column has {nan_count} NaN values.")

target_median = train['target'].median()
train['target'].fillna(target_median, inplace=True)

print(f"converting prices columns to float32 values.")
train = convert_price_cols_float32(train)
# ----------------------------- Reading train data -------------------------

The 'target' column has 88 NaN values.
converting prices columns to float32 values.


C:\Users\cwang\AppData\Local\Temp\ipykernel_44524\241488516.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['target'].fillna(target_median, inplace=True)


In [4]:
#@title functions

def split_by_date(df, dates):

    df_start, df_end = dates
    df = df[(df['date_id'] >= df_start) & (df['date_id'] <=df_end)].reset_index(drop=True)

    return df



def lag_function(df, columns_to_lag, numbers_of_days_to_lag):

    df_indexed = df.set_index(['stock_id', 'seconds_in_bucket', 'date_id'])
    
    for column_to_lag in columns_to_lag:
        for number_days_to_lag in numbers_of_days_to_lag:
            df_indexed[f'lag{number_days_to_lag}_{column_to_lag}'] = df_indexed.groupby(level=['stock_id', 'seconds_in_bucket'])[column_to_lag].shift(number_days_to_lag)
    
    df_indexed.reset_index(inplace=True)
    
    return df_indexed



def create_diff_lagged_features_within_date_revised(df, columns_to_lag, numbers_of_lag):
    df_copy = df.copy()

    # Store the new columns in a list
    new_columns = []

    # Iterate through each specified lag
    for lag in numbers_of_lag:
        # Create lagged dataframe once per lag value
        lagged_df = df.groupby(['stock_id', 'date_id'])[columns_to_lag].shift(periods=lag)
        
        # Iterate through each specified column
        for column in columns_to_lag:
            # Compute the new column
            new_col_name = f'{column}_diff_lag{lag}'
            new_column = df[column] - lagged_df[column]
            
            # Store the new column in the list
            new_columns.append(new_column.rename(new_col_name))

    # Concatenate the original dataframe with the new columns
    result_df = pd.concat([df_copy] + new_columns, axis=1)
    
    return result_df


def create_features_to_start_optimized(df, features_list):

    first_values_df = df.groupby(['stock_id', 'date_id'])[features_list].transform('first')

    for feature in features_list:
        feature_to_start_col_name = f'{feature}_to_start'
        df[feature_to_start_col_name] = df[feature] - first_values_df[feature]

    return df


def compute_imbalances(df_, columns, prefix = ''):
    """Computes the differences and imbalances for pairs of columns and stores them in the DataFrame."""
    df = df_.copy()
    for col1, col2 in combinations(columns, 2):
        
        # Sort the columns lexicographically to ensure consistent ordering
        col1, col2 = sorted([col1, col2])
        
        # Compute imbalance directly without creating a temporary difference column
        total = df[col1] + df[col2]
        imbalance_column_name = f'{col1}_{col2}_imb{prefix}'
        
        # Ensure we don't divide by zero
        df[imbalance_column_name] = (df[col1] - df[col2]).divide(total, fill_value=np.nan)

    return df

def compute_percentage_difference(df, columns, prefix = ''):

    df_copy = df.copy()
    
    # Iterate over all combinations of two different price columns
    for col1, col2 in combinations(columns, 2):
        # Sort the columns lexicographically to ensure consistent ordering
        col1, col2 = sorted([col1, col2])

        # Create a new column name based on the price columns
        new_col_name = f'pct_diff_{col1}_vs_{col2}_{prefix}'

        # Compute the percentage difference
        df_copy[new_col_name] = (df_copy[col1] - df_copy[col2]) / df_copy[col2] * 100

    return df_copy



def create_deviation_within_seconds(df, num_features):
    groupby_cols = ['date_id', 'seconds_in_bucket']
    new_columns = {}  # Dictionary to hold new columns

    for feature in num_features:
        grouped_median = df.groupby(groupby_cols)[feature].transform('median')
        deviation_col_name = f'deviation_from_median_{feature}'
        new_columns[deviation_col_name] = df[feature] - grouped_median

    # Concatenate all new columns at once
    df = pd.concat([df, pd.DataFrame(new_columns)], axis=1)
    return df


def create_cumsum_features(df, columns_to_compute):
    df_copy = df.copy()
    
    # Group by 'stock_id' and 'date_id' for cumulative sum calculation
    grouped = df_copy.groupby(['stock_id', 'date_id'])
    
    # Calculate cumulative sum for each column within each group
    for column in columns_to_compute:
        cumsum_col_name = f'{column}_cumsum'
        df_copy[cumsum_col_name] = grouped[column].cumsum()
    return df_copy


def save_pickle(data, file_path):
 
    # Create the directory if it doesn't exist
    directory = os.path.dirname(file_path)
    if not os.path.exists(directory):
        os.makedirs(directory)

    # Save the pickle file
    with open(file_path, 'wb') as file:
        pickle.dump(data, file)

    print(f"Data saved to {file_path}")
    #example: save_pickle(all_data, 'k8/all_data.pkl')

def load_pickle(file_path):

    # Load and return the data from the pickle file
    if os.path.exists(file_path):
        with open(file_path, 'rb') as file:
            data = pickle.load(file)
        return data
    else:
        raise FileNotFoundError(f"No such file: {file_path}")


In [5]:
#@title global
#---------------------------------- Global based on stock_id --------------------------------------

def aggregated_features_dic(df):
    global_feats = {}

    columns_to_aggregate= ['bid_size','ask_size']
    groupby_cols=['stock_id']

    def q25(x):
        return x.quantile(0.25)

    def q75(x):
        return x.quantile(0.75)

    # Define the aggregations
    aggregations = ['mean', 'median', 'std', 'min', 'max', q25, q75]

    # Prepare a dictionary to hold the aggregated Series


    # Perform aggregation for each column and operation
    for column in columns_to_aggregate:
        for agg in aggregations:
            # Define the aggregation function name
            if callable(agg):
                func_name = agg.__name__
            else:
                func_name = agg
            
            # Perform the aggregation
            agg_series = df.groupby(groupby_cols)[column].agg(agg)
            # Create a new feature name and add it to the dictionary
            new_feature_name = f"{func_name}_{column}"
            global_feats[new_feature_name] = agg_series

    global_feats["median_size"] = df.groupby("stock_id")["bid_size"].median() + df.groupby("stock_id")["ask_size"].median()
    global_feats["std_size"] = df.groupby("stock_id")["bid_size"].std() + df.groupby("stock_id")["ask_size"].std()
    global_feats["ptp_size"] = df.groupby("stock_id")["bid_size"].max() - df.groupby("stock_id")["bid_size"].min()
    global_feats["median_price"] = df.groupby("stock_id")["bid_price"].median() + df.groupby("stock_id")["ask_price"].median()
    global_feats["std_price"] = df.groupby("stock_id")["bid_price"].std() + df.groupby("stock_id")["ask_price"].std()
    global_feats["ptp_price"] = df.groupby("stock_id")["bid_price"].max() - df.groupby("stock_id")["ask_price"].min()


    return global_feats

aggregated_dic = aggregated_features_dic(train)

def map_global(df,dict):
    df_ = df.copy()
    for key, value in dict.items():
        df_[f"global_{key}"] = df_["stock_id"].map(value.to_dict())
    
    return df_


In [6]:
#@title helper functions

def flatten_outliers_y_train(y_train, lower_quantile=0.01, upper_quantile=0.99):

    lower_bound = np.quantile(y_train, lower_quantile)
    upper_bound = np.quantile(y_train, upper_quantile)

    # Cap values below the lower bound and above the upper bound
    y_train_flattened = np.clip(y_train, lower_bound, upper_bound)

    return y_train_flattened


def create_autocorrelation_features(df, columns, lags):

    df_copy = df.copy()
    
    for column in columns:
        for lag in lags:
            lagged_series = df_copy[column].shift(lag)
            df_copy[f'{column}_autocorr_lag{lag}'] = df_copy[column].corrwith(lagged_series)

    return df_copy


def calculate_stat_lag(df, num_lags):

    lags = [f'lag{i}_target' for i in range(1, num_lags + 1)]

    df['target_mean'] = df[lags].mean(axis=1)
    df['target_std_dev'] = df[lags].std(axis=1)
    df['target_variance'] = df[lags].var(axis=1)
    df['target_median'] = df[lags].median(axis=1)
    df['target_range'] = df[lags].max(axis=1) - df[lags].min(axis=1)

    return df


def calculate_stat(df, cols, prefix='prices'):

    df[f'{prefix}_mean'] = df[cols].mean(axis=1)
    df[f'{prefix}_std_dev'] = df[cols].std(axis=1)
    df[f'{prefix}_variance'] = df[cols].var(axis=1)
    df[f'{prefix}_median'] = df[cols].median(axis=1)
    df[f'{prefix}_range'] = df[cols].max(axis=1) - df[cols].min(axis=1)

    return df

In [7]:
def make_predictions(models, X_test,model = 'nn'):
    if model == 'nn':
        all_predictions = [model.predict(X_test, batch_size=16384) for model in models]
    if model == 'lgb' or model == 'xgb' or model == 'cat':
        all_predictions = [model.predict(X_test) for model in models]
    prediction = np.mean(all_predictions, axis=0)
    return prediction

In [8]:
#@title pipeline

raw_cols          = ['imbalance_size','matched_size','bid_size','ask_size','reference_price','far_price','near_price','bid_price','ask_price','wap','imbalance_buy_sell_flag'] 

columns_prices    = ['reference_price','far_price','near_price','bid_price','ask_price','wap']
columns_4prices   = ['reference_price','bid_price','ask_price','wap']

columns_sizes     = ['imbalance_size','matched_size','bid_size','ask_size']
columns_flag      = ['imbalance_buy_sell_flag'] 


diff_lags           = [1, 2, 3, 6, 12, 18, 24]
diff_lags_extra     = [30, 36, 42, 48]

num_of_target_lags  = 12
target_lags         = list(range(1,num_of_target_lags+1))


def feature_engineering(df):
    
    df = df.copy()

    df['spread_eng']                            = df['ask_price'] - df['bid_price'] 
    df['volume_eng']                            = df['bid_size'] + df['ask_size']  
    df['volumne_imbalance_eng']                 = df['bid_size'] - df['ask_size']  
    
    df['imbalance_ratio']                       = df['imbalance_size'] / df['matched_size']  #(RM) Good

    df['price_spread_near_far']                 = df['near_price'] - df['far_price']   #RM (debatable)
    df['price_wap_difference_eng']              = df['reference_price'] - df['wap']    

    df['weighted_imbalance_eng']                = df['imbalance_size'] * df['imbalance_buy_sell_flag'] #very important


    df['bid_ask_ratio']                         = df['bid_size'] / df['ask_size'] #(RM) neutral
    df['imbalance_to_bid_ratio_eng']            = df['imbalance_size'] / df['bid_size']
    df['imbalance_to_ask_ratio_eng']            = df['imbalance_size'] / df['ask_size']
    df['matched_size_to_total_size_ratio_eng']  = df['matched_size'] / (df['bid_size'] + df['ask_size'])


    return df


def feature_pipeline(df):
    
    if df.empty:
        return pd.DataFrame()
        
    #--------------- connected ------------
    df = feature_engineering(df)
    df = compute_imbalances(df, columns_sizes,prefix='_sz_')
    df = compute_imbalances(df, columns_prices,prefix = '_pr_')



    eng_features       = [feature for feature in df.columns if "_eng" in feature]
    imb_features_all   = [feature for feature in df.columns if "_imb_" in feature]
    imb_features_price = [feature for feature in df.columns if "_pr_" in feature]
    imb_features_size  = [feature for feature in df.columns if "_sz_" in feature]

    #--------------- connected ------------


    diff_lag_cols = raw_cols + eng_features
    print(f"diff lagging {len(diff_lag_cols)} columns for {len(diff_lags)} lags.")
    df = create_diff_lagged_features_within_date_revised(df,diff_lag_cols,diff_lags)

    cumsum_columns = columns_sizes + imb_features_size + eng_features 
    print(f"cumsum for {len(cumsum_columns)} cols.")
    df = create_cumsum_features(df, cumsum_columns)

    deviation_cols = raw_cols + eng_features + imb_features_size #+ imb_features_price
    print(f"deviation {len(deviation_cols)} columns within seconds.")
    df = create_deviation_within_seconds(df,deviation_cols)

    print(f"lagging target column for {len(target_lags)} lags.")
    df = lag_function(df, ['target'], target_lags)

    df = map_global(df,aggregated_dic)

    df = calculate_stat_lag(df, num_lags=num_of_target_lags)

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    print("Done...")
    
    return df

In [9]:
#@title runnig pipeline

excluded_columns = ['row_id', 'date_id', 'time_id', 'target','stock_return']  

train_eng = feature_pipeline(train)

lgb_features = [col for col in train_eng.columns if col not in excluded_columns]
categorical_features = ['seconds_in_bucket']

print("we have {} lgb features".format(len(lgb_features)))
    
train_data       = split_by_date(train_eng, dates_train)
test_data        = split_by_date(train_eng, dates_test)
print("number of dates in train = {} , number of dates in test {}".format (train_data['date_id'].nunique(),test_data['date_id'].nunique()))

cleaning = False
if cleaning:
    import gc
    #del train
    del train_eng
    gc.collect()

diff lagging 19 columns for 7 lags.
cumsum for 18 cols.
deviation 25 columns within seconds.
lagging target column for 12 lags.
Done...
we have 258 lgb features
number of dates in train = 481 , number of dates in test 0


In [10]:
#@title TPU
try:
    # Create a TPUClusterResolver
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    # Connect to the TPU cluster
    tf.config.experimental_connect_to_cluster(tpu)
    # Initialize the TPU system
    tf.tpu.experimental.initialize_tpu_system(tpu)
    # Create a TPUStrategy for distributed training
    tpu_strategy = tf.distribute.experimental.TPUStrategy(tpu)
except ValueError:
    tpu_strategy = None  # No TPU found

In [11]:
print("--------------test data is empty so adjusting the last date for test-----------------")
test_data = train_data.query("date_id == 480").copy()


X_train, y_train = train_data[lgb_features], train_data['target']
X_test, y_test = test_data[lgb_features],test_data['target']
train_set = lgb.Dataset(X_train, label=y_train,categorical_feature=categorical_features,free_raw_data=False)
test_set = lgb.Dataset(X_test, label=y_test,categorical_feature=categorical_features,free_raw_data=False)

--------------test data is empty so adjusting the last date for test-----------------


In [19]:
directory = os.path.dirname(models_path)
if not os.path.exists(directory):
    os.makedirs(directory)
            
lgb_models = []
for i in range(num_models['lgb']):
    rnd_state=42+i
    print(f"Training model {i+1} out of {num_models['lgb']} with seed {rnd_state}")
    print("---------------------------------------")

    lgb_params['random_state'] = rnd_state
    lgb_model = lgb.train(lgb_params, train_set,init_model=None, valid_sets=[train_set, test_set],
                            callbacks=[lgb.early_stopping(stopping_rounds=250), lgb.log_evaluation(250)])
    # lgb_model = lgb.train(lgb_params, train_set,init_model=None, valid_sets=[X_test, y_test],
    #                         callbacks=[lgb.early_stopping(stopping_rounds=250), lgb.log_evaluation(250)])

    lgb_model.save_model(f'{models_path}model_lgb_{i}.txt')
    lgb_models.append(lgb_model)
            
    if dates_train[1]!=480:
        pred = lgb_model.predict(X_test)
        mae = mean_absolute_error(test_data['target'] , pred)
        print(f"Mean Absolute Error on test data: {mae:.5f}")

if dates_train[1]!=480:
    predictions =  make_predictions(lgb_models, test_data[lgb_features],model = 'lgb')                   
    print(f"LGB Ensemble Mean Absolute Error: {mean_absolute_error(test_data['target'], predictions):.5f}")

Training model 1 out of 1 with seed 42
---------------------------------------


c:\Users\cwang\anaconda3\envs\work\Lib\site-packages\lightgbm\engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


KeyboardInterrupt: 